In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [2]:
df = pd.read_csv("data_processed.csv")
df = df.drop("Unnamed: 0", axis=1)
df = df.drop("camera_id", axis=1)
df.head()

,age_days,daily_usage_hours,avg_temperature_c,avg_humidity_percent,vibration_level,signal_strength_dbm,avg_power_consumption_w,firmware_version,camera_type,location_type,...,old_camera,high_temp,high_humidity,weak_signal,high_usage,power_per_hour,temp_humidity,age_vibration,signal_quality,risk_score
0,119.0,16.500000,33.2,68.8,0.39,-55.3,12.0,7,PTZ,Outdoor_Commercial,...,0,0,0,0,0,0.727273,2284.16,0.127151,44.7,0
1,678.0,17.083333,16.9,73.6,0.55,-57.8,14.0,8,Bullet,Warehouse,...,0,0,0,0,0,0.819512,1243.84,1.021644,42.2,0
2,552.0,17.600000,22.6,72.2,0.05,-52.0,13.6,3,Thermal,Indoor,...,0,0,0,0,0,0.772727,1631.72,0.075616,48.0,0
3,1101.5,17.216667,25.9,77.5,0.45,-52.8,16.7,10,Bullet,Indoor,...,1,0,0,0,0,0.969990,2007.25,1.358014,47.2,1
4,339.0,17.350000,23.8,92.2,0.29,-52.8,14.9,7,Bullet,Indoor,...,0,0,1,0,0,0.858790,2194.36,0.269342,47.2,1


In [3]:
X = df.drop('needs_maintenance',axis=1)
y = df['needs_maintenance']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

In [5]:
catcols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numcols = X.select_dtypes(
    exclude=["object", "category"]
).columns.tolist()


In [6]:
Numpipe = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])
Numpipelog = Pipeline([
    ('imp', SimpleImputer(strategy='median'))
])
Catpipe = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('enc', OneHotEncoder(handle_unknown='ignore'))
])

In [7]:
prep = ColumnTransformer(
    transformers = [
        ('num', Numpipe, numcols),
        ('cat', Catpipe, catcols)
    ]
)

prep1 = ColumnTransformer(
    transformers = [
        ('num', Numpipelog, numcols),
        ('cat', Catpipe, catcols)
    ]
)

In [8]:
model_1 = ImbPipeline([
    ('prep', prep1),
    ('smote', SMOTE(random_state=42)),
    ('model', LogisticRegression(max_iter=1000))
])

In [9]:
model_1.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imp',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['age_days',
                                                   'daily_usage_hours',
                                                   'avg_temperature_c',
                                                   'avg_humidity_percent',
                                                   'vibration_level',
                                                   'signal_strength_dbm',
                                                   'avg_power_consumption_w',
                                                   'firmware_version',
                                                   'camera_age_years',
                                                   'old_camera', 'high_temp',
                

In [10]:
lr_predictions = model_1.predict(X_test)
lr_probability = model_1.predict_proba(X_test)[:,1]

In [11]:
model_2 = ImbPipeline([
    ("prep", prep),
    ("smote", SMOTE(random_state=42)),
    ("model", RandomForestClassifier(
       n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
    ))
])

In [12]:
model_2.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imp',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['age_days',
                                                   'daily_usage_hours',
                                                   'avg_temperature_c',
                                                   'avg_humidity_percent',
                                                   'vibration_level',
                                                   'signal_strength_dbm',
                                                   'avg_power_consumption_w',
                                                   'firmware_version',
   

In [13]:
rf_predictions = model_2.predict(X_test)
rf_probability = model_2.predict_proba(X_test)[:,1]

In [14]:
xgb_pipeline = ImbPipeline([
    ("prep", prep),
    ("smote", SMOTE(random_state=42)),
    ("model",
        XGBClassifier(
            n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
        )
    )
])

In [15]:
xgb_pipeline.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imp',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['age_days',
                                                   'daily_usage_hours',
                                                   'avg_temperature_c',
                                                   'avg_humidity_percent',
                                                   'vibration_level',
                                                   'signal_strength_dbm',
                                                   'avg_power_consumption_w',
                                                   'firmware_version',
   

In [16]:
xgb_predictions = xgb_pipeline.predict(X_test)
xgb_probability = xgb_pipeline.predict_proba(X_test)[:,1]

In [17]:
#MODEL SAVING
import joblib

['C:\\Users\\AiK\\Machine Learning Projects\\CCTV Maintanence\\Model Building\\Models\\xgboost_pipeline.pkl']

In [19]:
print(model_1)

print(model_2)

print(xgb_pipeline)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imp',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['age_days',
                                                   'daily_usage_hours',
                                                   'avg_temperature_c',
                                                   'avg_humidity_percent',
                                                   'vibration_level',
                                                   'signal_strength_dbm',
                                                   'avg_power_consumption_w',
                                                   'firmware_version',
                                                   'camera_age_years',
                                                   'old_camera', 'high_temp',
                

In [20]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score
)

In [21]:
def evaluate_model(model, X_test, y_test):

    y_pred = model.predict(X_test)

    y_prob = model.predict_proba(X_test)[:,1]

    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(y_test, y_pred)

    recall = recall_score(y_test, y_pred)

    f1 = f1_score(y_test, y_pred)

    roc = roc_auc_score(y_test, y_prob)
    
    print(f"Accuracy : {accuracy:.4f}")

    print(f"Precision : {precision:.4f}")

    print(f"Recall : {recall:.4f}")

    print(f"F1 : {f1:.4f}")

    print(f"ROC AUC : {roc:.4f}")

    print()

    print(classification_report(y_test,y_pred))

    return accuracy,precision,recall,f1,roc

In [22]:
lr_metrics = evaluate_model(
    model_1,
    X_test,
    y_test
)
rf_metrics = evaluate_model(
    model_2,
    X_test,
    y_test
)
xgb_metrics = evaluate_model(
    xgb_pipeline,
    X_test,
    y_test
)

Accuracy : 0.7836
Precision : 0.3074
Recall : 0.8329
F1 : 0.4491
ROC AUC : 0.8870

              precision    recall  f1-score   support

           0       0.98      0.78      0.87      8941
           1       0.31      0.83      0.45      1059

    accuracy                           0.78     10000
   macro avg       0.64      0.81      0.66     10000
weighted avg       0.90      0.78      0.82     10000

Accuracy : 0.8806
Precision : 0.4386
Recall : 0.4551
F1 : 0.4467
ROC AUC : 0.8699

              precision    recall  f1-score   support

           0       0.94      0.93      0.93      8941
           1       0.44      0.46      0.45      1059

    accuracy                           0.88     10000
   macro avg       0.69      0.69      0.69     10000
weighted avg       0.88      0.88      0.88     10000

Accuracy : 0.8948
Precision : 0.5046
Recall : 0.3645
F1 : 0.4232
ROC AUC : 0.8736

              precision    recall  f1-score   support

           0       0.93      0.96      0.9

In [23]:
joblib.dump(
    model_1,
    r"C:\Users\AiK\Machine Learning Projects\CCTV Maintanence\Model Building\Models\logistic_pipeline.pkl"
)

joblib.dump(
    model_2,
    r"C:\Users\AiK\Machine Learning Projects\CCTV Maintanence\Model Building\Models\random_forest_pipeline.pkl"
)

joblib.dump(
    xgb_pipeline,
    r"C:\Users\AiK\Machine Learning Projects\CCTV Maintanence\Model Building\Models\xgboost_pipeline.pkl"
)

['C:\\Users\\AiK\\Machine Learning Projects\\CCTV Maintanence\\Model Building\\Models\\xgboost_pipeline.pkl']

In [24]:
df.to_csv(r"C:\Users\AiK\Machine Learning Projects\CCTV Maintanence\Predictions\final.csv")

In [25]:
evaluation_df = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        lr_metrics[0],
        rf_metrics[0],
        xgb_metrics[0]
    ],
    "Precision": [
        lr_metrics[1],
        rf_metrics[1],
        xgb_metrics[1]
    ],
    "Recall": [
        lr_metrics[2],
        rf_metrics[2],
        xgb_metrics[2]
    ],
    "F1 Score": [
        lr_metrics[3],
        rf_metrics[3],
        xgb_metrics[3]
    ],
    "ROC AUC": [
        lr_metrics[4],
        rf_metrics[4],
        xgb_metrics[4]
    ]
})

evaluation_df.to_csv("evaluation_metrics.csv", index=False)